# PathSnap Model Training & Benchmark: Random Forest Classifier (Bagging Ensemble)
**Course:** UE24CS352A - Machine Learning (PES University)  
**Team 6:** PathSnap Distributed Map-Matching  
**Model Name:** `random_forest`  

---
### Notebook Objectives
1. Inspect hyperparameters, training pipeline, and environment provenance.
2. Review 5-fold group-aware cross-validation on 80 training trips (16 trips/fold).
3. Evaluate held-out generalization on the 10 student-test trips.
4. Verify observation-level Top-1 accuracy, Macro F1, and categorical log loss.


In [1]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, HTML, Image

PROJ_ROOT = Path("..").resolve()
MODEL_DIR = PROJ_ROOT / "results" / "run_01" / "random_forest"
sys.path.insert(0, str(PROJ_ROOT / "scripts"))

print("Model output directory:", MODEL_DIR)


Model output directory: C:\Users\joshwin\PESU\3RD YEAR\5th sem\ML\team_6\results\run_01\random_forest


## 1. Model Configuration & Provenance
Inspection of hyperparameters, random seeds, and package dependencies.


In [2]:
with open(MODEL_DIR / "configuration.json") as f:
    config = json.load(f)
with open(MODEL_DIR / "provenance.json") as f:
    prov = json.load(f)

print(f"Algorithm: {prov['model']} (Random Seed: {prov['seed']})")
print(f"Scikit-Learn Version: {prov['sklearn']} | Python: {prov['python']}")
print("\nKey Parameters:")
for k in sorted(config.keys())[:12]:
    print(f"  {k}: {config[k]}")


Algorithm: random_forest (Random Seed: 212)
Scikit-Learn Version: 1.9.1 | Python: 3.12.6

Key Parameters:
  bootstrap: True
  ccp_alpha: 0.0
  class_weight: None
  criterion: gini
  max_depth: 20
  max_features: sqrt
  max_leaf_nodes: None
  max_samples: None
  min_impurity_decrease: 0.0
  min_samples_leaf: 2
  min_samples_split: 2
  min_weight_fraction_leaf: 0.0


## 2. 5-Fold Cross-Validation & Student-Test Evaluation
Group-aware 5-fold evaluation across all 80 training trips, followed by refitting on the full training set and testing on the 10 student-test trips.


In [3]:
with open(MODEL_DIR / "metrics.json") as f:
    records = json.load(f)

rows = []
for r in records:
    m = r["metrics"]["all_points"]
    rel = r["metrics"]["reliable_points"]
    rows.append({
        "Split": r["split"],
        "Top-1 Accuracy": f"{m['top1_accuracy']*100:.2f}%",
        "Macro F1": f"{m['macro_f1']:.4f}",
        "Macro Precision": f"{m['macro_precision']:.4f}",
        "Macro Recall": f"{m['macro_recall']:.4f}",
        "Log Loss": f"{m['categorical_log_loss_with_miss_state']:.4f}",
        "Reliable Top-1": f"{rel['top1_accuracy']*100:.2f}%",
        "Fit Time (s)": f"{r['fit_seconds']:.1f}s",
        "Inf Time (ms)": f"{r['inference_seconds']*1000:.1f}ms"
    })

df_metrics = pd.DataFrame(rows)
display(df_metrics)


,Split,Top-1 Accuracy,Macro F1,Macro Precision,Macro Recall,Log Loss,Reliable Top-1,Fit Time (s),Inf Time (ms)
0,fold_0,77.44%,0.6088,0.6474,0.5924,1.0743,86.03%,145.7s,564.1ms
1,fold_1,75.28%,0.6907,0.7216,0.6769,3.3079,87.39%,77.8s,870.4ms
2,fold_2,46.72%,0.5920,0.6336,0.5725,10.3636,81.21%,81.5s,1017.4ms
3,fold_3,74.79%,0.6523,0.6923,0.6324,0.8234,77.51%,63.2s,1400.0ms
4,fold_4,85.44%,0.7127,0.7425,0.7006,0.6510,91.63%,108.6s,591.9ms
5,student_test,81.06%,0.6215,0.6639,0.6053,0.7401,89.00%,87.0s,396.1ms


## 3. Performance Summary
Mean cross-validation metrics across all 5 folds.


In [4]:
with open(MODEL_DIR / "summary.json") as f:
    summary = json.load(f)

print(f"Completed Folds: {summary['completed_folds']}")
print(f"Mean CV Macro F1: {summary['cv_macro_f1_mean']:.4f} ± {summary['cv_macro_f1_std']:.4f}")
print(f"Mean CV Top-1 Accuracy: {summary['cv_top1_acc_mean']*100:.2f}%")
print(f"Student-Test Evaluation Complete: {summary['student_test_completed']}")


Completed Folds: 5
Mean CV Macro F1: 0.6513 ± 0.0461
Mean CV Top-1 Accuracy: 71.93%
Student-Test Evaluation Complete: True
